In [1]:
"""
Ambiguity Classification MLP
=============================

Architecture:
    Input (17) -> Shared [256->512->1024->512]
               -> NUM_AMBIGUITIES parallel branches, each: [256->128->C_i]

    C_i = number of integer classes for ambiguity i
        = (max_i - min_i + 1), computed from the training set

Pipeline:
    1. Load X, Y, ambiguity MAT files
    2. Scale Y with StandardScaler
    3. Train/val split  4/5 : 1/5
    4. Train with CrossEntropy per branch, summed
    5. Report per-branch accuracy + overall accuracy on val set
    6. Use predicted ambiguities to run the positioning solver on val set
       and report positioning error vs. using true ambiguities
    7. Save model weights + scaler; zip everything for Colab
"""

from __future__ import annotations

import os
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Optional, Sequence

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from scipy.io import loadmat, savemat
from scipy.optimize import minimize
from sklearn.preprocessing import StandardScaler
import joblib


# ============================================================
# User configuration
# ============================================================
X_FILE   = r"X_ngp_5_0_dbm.mat" # Should be modified according to the transmit power and GP training samples
Y_FILE   = r"Y_gp_ngp_5_0_dbm.mat" # Should be modified according to the transmit power and GP training samples
AMB_FILE = r"ambiguities.mat"

OUTPUT_DIR  = Path(".")
MODEL_FILE  = "ambiguity_mlp.pt"
SCALER_FILE = "y_scaler.pkl"
ZIP_FILE    = "model_and_scaler_0db_GP5.zip" # Should be modified according to the transmit power and GP training samples

AP_LOCS = np.array([
    [5.0000,4.5000,2.0000,6.5000,3.0000,5.0000,8.0000,8.0000,7.0000,1.2000,2.3000,7.5000,8.0000,1.3000,0.5000,8.0000,7.0000],
    [5.0000,3.0000,3.7300,3.0000,6.0000,7.0000,6.0000,4.0000,7.0000,3.0000,1.7300,1.0000,2.1500,7.0000,8.9000,9.2000,7.9000],
    [4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000,4.0000],
], dtype=float)

C    = 3e8
FREQ = 0.8e9

# Training hyper-parameters
BATCH_SIZE    = 512
NUM_EPOCHS    = 60
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
LR_STEP_SIZE  = 20
LR_GAMMA      = 0.3
DROPOUT_RATE  = 0.2

VAL_FRACTION  = 0.01
RANDOM_SEED   = 42

# Subsampling: set to None to use all data, or e.g. 50_000
NUM_SAMPLES   = 1000000

# Number of ambiguities to estimate (1–16).
# The first NUM_AMBIGUITIES differential ambiguities are used.
# For positioning, predicted values fill the first NUM_AMBIGUITIES rows;
# true values are used for the remaining (16 - NUM_AMBIGUITIES) rows.
NUM_AMBIGUITIES = 16

# Positioning solver settings
MAXITER = 300
TOL     = 1e-9

# ============================================================
# Reproducibility
# ============================================================
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# ============================================================
# MAT loading helpers
# ============================================================
def _is_numeric_array(x: Any) -> bool:
    return isinstance(x, np.ndarray) and np.issubdtype(x.dtype, np.number)


def _recursive_find_named(obj: Any, candidate_names: Sequence[str]) -> Optional[np.ndarray]:
    candidate_names = tuple(candidate_names)
    if isinstance(obj, dict):
        for name in candidate_names:
            if name in obj:
                value = obj[name]
                if _is_numeric_array(value):
                    return np.asarray(value, dtype=float)
                nested = _recursive_find_named(value, candidate_names)
                if nested is not None:
                    return nested
        for value in obj.values():
            nested = _recursive_find_named(value, candidate_names)
            if nested is not None:
                return nested
        return None
    if hasattr(obj, "_fieldnames"):
        for name in candidate_names:
            if name in getattr(obj, "_fieldnames", []):
                value = getattr(obj, name)
                if _is_numeric_array(value):
                    return np.asarray(value, dtype=float)
                nested = _recursive_find_named(value, candidate_names)
                if nested is not None:
                    return nested
        for field in getattr(obj, "_fieldnames", []):
            nested = _recursive_find_named(getattr(obj, field), candidate_names)
            if nested is not None:
                return nested
        return None
    if isinstance(obj, np.ndarray):
        if obj.dtype == object:
            for item in obj.flat:
                nested = _recursive_find_named(item, candidate_names)
                if nested is not None:
                    return nested
        return None
    return None


def load_mat_variable(mat_path: str | Path, candidate_names: Sequence[str]) -> np.ndarray:
    mat_path = str(mat_path)
    try:
        data = loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    except NotImplementedError as exc:
        raise RuntimeError(
            f"Could not read '{mat_path}'. It may be MATLAB v7.3. "
            "Please resave with -v7 or -v7.2."
        ) from exc
    clean = {k: v for k, v in data.items() if not k.startswith("__")}
    result = _recursive_find_named(clean, candidate_names)
    if result is None:
        available = ", ".join(clean.keys())
        raise KeyError(
            f"Could not find any of {candidate_names} in '{mat_path}'. "
            f"Top-level keys: {available}"
        )
    return np.asarray(result, dtype=float)


def ensure_2d(a: np.ndarray, name: str) -> np.ndarray:
    a = np.asarray(a, dtype=float)
    if a.ndim != 2:
        raise ValueError(f"{name} must be 2-D, but got shape {a.shape}")
    return a


# ============================================================
# Phase preprocessing
# ============================================================
def preprocess_y_phases_correct(y_raw: np.ndarray):
    y_raw = ensure_2d(y_raw, "Y")
    y_mod = np.mod(y_raw, 2.0 * np.pi)
    phase_diff_cycles = (y_mod[1:, :] - y_mod[0:1, :]) / (2.0 * np.pi)
    return y_mod, phase_diff_cycles


def preprocess_ambiguities(amb_raw: np.ndarray) -> np.ndarray:
    """Always returns full (16, N) differential ambiguity matrix."""
    amb_raw = ensure_2d(amb_raw, "ambiguity")
    if amb_raw.shape[0] == 16:
        return np.asarray(amb_raw, dtype=float)
    if amb_raw.shape[0] == 17:
        return np.asarray(amb_raw[1:, :] - amb_raw[0:1, :], dtype=float)
    raise ValueError(f"Ambiguity matrix must have 16 or 17 rows, got {amb_raw.shape}")


# ============================================================
# Positioning helpers
# ============================================================
def check_ap_locs(ap_locs: np.ndarray) -> np.ndarray:
    ap_locs = ensure_2d(ap_locs, "ap_locs")
    if ap_locs.shape != (3, 17):
        raise ValueError(f"ap_locs must be (3,17), got {ap_locs.shape}")
    return ap_locs


def dr_model(x: np.ndarray, ap_locs: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float).reshape(3)
    d_all = np.linalg.norm(ap_locs - x[:, None], axis=0)
    return d_all[1:] - d_all[0]


def dr_jacobian(x: np.ndarray, ap_locs: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    x = np.asarray(x, dtype=float).reshape(3)
    diff = x[:, None] - ap_locs
    d_all = np.maximum(np.linalg.norm(diff, axis=0), eps)
    grad_all = (diff / d_all[None, :]).T
    return grad_all[1:, :] - grad_all[0:1, :]


def objective_and_grad(x, meas_dr, ap_locs):
    residual = dr_model(x, ap_locs) - meas_dr
    jac = dr_jacobian(x, ap_locs)
    return 0.5 * float(np.dot(residual, residual)), jac.T @ residual


def estimate_single_position(meas_dr, ap_locs, x0=None):
    if x0 is None:
        x0 = np.mean(ap_locs, axis=1)
    result = minimize(
        fun=lambda x: objective_and_grad(x, meas_dr, ap_locs)[0],
        x0=x0,
        jac=lambda x: objective_and_grad(x, meas_dr, ap_locs)[1],
        method="L-BFGS-B",
        options={"maxiter": MAXITER, "ftol": TOL, "gtol": TOL},
    )
    return result.x.astype(float)


def estimate_positions_from_delta_r(delta_r: np.ndarray, ap_locs: np.ndarray,
                                     x_init: Optional[np.ndarray] = None) -> np.ndarray:
    """Estimate (3, N) positions from (16, N) differential ranges."""
    n = delta_r.shape[1]
    x_hat = np.zeros((3, n), dtype=float)
    x0 = np.mean(ap_locs, axis=1) if x_init is None else x_init[:, 0].copy()
    for i in range(n):
        xi0 = x_init[:, i] if x_init is not None else x0
        x_hat[:, i] = estimate_single_position(delta_r[:, i], ap_locs, xi0)
        if x_init is None:
            x0 = x_hat[:, i]
        if (i + 1) % max(1, n // 10) == 0 or i == n - 1:
            print(f"  Positioned {i+1}/{n}")
    return x_hat


def combine_phase_and_ambiguity(amb_diff, phase_diff_cycles):
    wavelength = C / FREQ
    return wavelength * (amb_diff + phase_diff_cycles), wavelength


# ============================================================
# Label encoding per ambiguity
# ============================================================
@dataclass
class AmbiguityLabel:
    """Maps integer ambiguity values <-> class indices for one branch."""
    amb_min: int
    amb_max: int

    @property
    def num_classes(self) -> int:
        return self.amb_max - self.amb_min + 1

    def to_class(self, values: np.ndarray) -> np.ndarray:
        idx = np.round(values).astype(int) - self.amb_min
        idx = np.clip(idx, 0, self.num_classes - 1)
        return idx.astype(np.int64)

    def to_ambiguity(self, class_idx: np.ndarray) -> np.ndarray:
        return class_idx.astype(float) + self.amb_min


def build_label_encoders(amb_diff_train: np.ndarray) -> list[AmbiguityLabel]:
    """Compute per-branch label range from training data. Shape: (num_amb, n_train)."""
    encoders = []
    for i in range(amb_diff_train.shape[0]):
        lo = int(np.floor(amb_diff_train[i].min()))
        hi = int(np.ceil(amb_diff_train[i].max()))
        encoders.append(AmbiguityLabel(amb_min=lo, amb_max=hi))
    return encoders


# ============================================================
# Model definition
# ============================================================
class AmbiguityMLP(nn.Module):
    """
    Shared trunk:  17 -> 256 -> 512 -> 1024 -> 512
    NUM_AMBIGUITIES branches:  512 -> 256 -> 128 -> C_i
    """

    def __init__(self, num_classes_per_branch: list[int], dropout: float = DROPOUT_RATE):
        super().__init__()
        self.num_branches = len(num_classes_per_branch)

        def block(in_f, out_f):
            return nn.Sequential(
                nn.Linear(in_f, out_f),
                nn.BatchNorm1d(out_f),
                nn.ReLU(inplace=True),
                nn.Dropout(dropout),
            )

        self.shared = nn.Sequential(
            block(17,   256),
            block(256,  512),
            block(512,  1024),
            block(1024, 512),
        )

        self.branches = nn.ModuleList()
        for c in num_classes_per_branch:
            branch = nn.Sequential(
                block(512, 256),
                block(256, 128),
                nn.Linear(128, c),
            )
            self.branches.append(branch)

    def forward(self, x: torch.Tensor) -> list[torch.Tensor]:
        shared_out = self.shared(x)
        return [branch(shared_out) for branch in self.branches]


# ============================================================
# Training & evaluation
# ============================================================
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x_batch, *y_batches in loader:
        x_batch   = x_batch.to(device)
        y_batches = [y.to(device) for y in y_batches]
        optimizer.zero_grad()
        logits_list = model(x_batch)
        loss = sum(criterion(logits_list[i], y_batches[i]) for i in range(len(logits_list)))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x_batch.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion, device, encoders: list[AmbiguityLabel]):
    num_amb = len(encoders)
    model.eval()
    total_loss = 0.0
    all_preds  = [[] for _ in range(num_amb)]
    all_labels = [[] for _ in range(num_amb)]

    for x_batch, *y_batches in loader:
        x_batch   = x_batch.to(device)
        y_batches = [y.to(device) for y in y_batches]
        logits_list = model(x_batch)
        loss = sum(criterion(logits_list[i], y_batches[i]) for i in range(num_amb))
        total_loss += loss.item() * x_batch.size(0)
        for i in range(num_amb):
            all_preds[i].append(logits_list[i].argmax(dim=1).cpu().numpy())
            all_labels[i].append(y_batches[i].cpu().numpy())

    all_preds  = [np.concatenate(p) for p in all_preds]
    all_labels = [np.concatenate(l) for l in all_labels]

    branch_acc  = np.array([float(np.mean(all_preds[i] == all_labels[i])) for i in range(num_amb)])
    overall_acc = float(np.mean([all_preds[i] == all_labels[i] for i in range(num_amb)]))
    avg_loss    = total_loss / len(loader.dataset)
    return avg_loss, branch_acc, overall_acc, all_preds


# ============================================================
# Plotting helpers
# ============================================================
def plot_training_curves(train_losses, out_dir):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_losses, label="Train loss", linewidth=2)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Training cross-entropy loss")
    ax.legend(); ax.grid(True)
    fig.tight_layout()
    path = out_dir / "training_curves.png"
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved training curves -> {path}")


def plot_branch_accuracy(branch_acc: np.ndarray, out_dir: Path):
    num_amb = len(branch_acc)
    fig, ax = plt.subplots(figsize=(max(6, num_amb * 0.7), 4))
    ax.bar(np.arange(1, num_amb + 1), branch_acc * 100)
    ax.set_xlabel("Ambiguity branch index"); ax.set_ylabel("Accuracy [%]")
    ax.set_title(f"Per-branch classification accuracy (validation set, {num_amb} branches)")
    ax.set_xticks(np.arange(1, num_amb + 1)); ax.grid(axis="y")
    fig.tight_layout()
    path = out_dir / "branch_accuracy.png"
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved branch accuracy -> {path}")


def plot_position_error_cdf(errors_true_init, errors_pred, out_dir: Path):
    fig, ax = plt.subplots(figsize=(7, 5))
    for errors, label in [(errors_true_init, "True init (reference)"),
                          (errors_pred,       "Predicted ambiguities")]:
        e = np.sort(np.maximum(errors, 1e-12))
        cdf = np.arange(1, e.size + 1) / e.size
        ax.plot(e, cdf, linewidth=2, label=label)
    ax.set_xscale("log"); ax.set_xlabel("Position error [m]"); ax.set_ylabel("CDF")
    ax.set_title("Positioning error CDF (validation set)")
    ax.legend(); ax.grid(True, which="both", linestyle="--", alpha=0.6)
    fig.tight_layout()
    path = out_dir / "position_error_cdf.png"
    fig.savefig(str(path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved position error CDF -> {path}")


# ============================================================
# Main
# ============================================================
def main():
    import time

    if not (1 <= NUM_AMBIGUITIES <= 16):
        raise ValueError(f"NUM_AMBIGUITIES must be between 1 and 16, got {NUM_AMBIGUITIES}")

    out_dir = OUTPUT_DIR
    out_dir.mkdir(parents=True, exist_ok=True)

    ap_locs = check_ap_locs(np.asarray(AP_LOCS, dtype=float))

    # ----------------------------------------------------------
    # 1. Load data
    # ----------------------------------------------------------
    print("Loading data...")
    x_true  = load_mat_variable(X_FILE,   ("X",))
    y_raw   = load_mat_variable(Y_FILE,   ("Y_theoretical", "Y_gp"))
    amb_raw = load_mat_variable(AMB_FILE, ("diff_ambiguities", "ambiguities", "z", "result_matrix"))

    x_true  = ensure_2d(x_true,  "X")           # (3, N)
    y_raw   = ensure_2d(y_raw,   "Y")            # (17, N)
    amb_raw = ensure_2d(amb_raw, "ambiguity")    # (16 or 17, N)

    N = x_true.shape[1]
    print(f"Total samples in files: {N}")

    # Subsample
    if NUM_SAMPLES is not None and NUM_SAMPLES < N:
        rng_sub = np.random.default_rng(RANDOM_SEED)
        sub_idx = rng_sub.choice(N, size=NUM_SAMPLES, replace=False)
        sub_idx.sort()
        x_true  = x_true[:,  sub_idx]
        y_raw   = y_raw[:,   sub_idx]
        amb_raw = amb_raw[:, sub_idx]
        N = NUM_SAMPLES
        print(f"Subsampled to {N} samples (NUM_SAMPLES={NUM_SAMPLES})")
    else:
        print(f"Using all {N} samples")

    # Phase preprocessing
    y_mod, phase_diff_cycles = preprocess_y_phases_correct(y_raw)  # (17,N), (16,N)
    amb_diff_full = preprocess_ambiguities(amb_raw)                  # (16, N) full matrix

    # Keep only the first NUM_AMBIGUITIES rows for training
    amb_diff = amb_diff_full[:NUM_AMBIGUITIES, :]                    # (NUM_AMBIGUITIES, N)
    print(f"\nEstimating {NUM_AMBIGUITIES}/16 ambiguities")

    # ----------------------------------------------------------
    # 2. Train / val split  (4/5 : 1/5)
    # ----------------------------------------------------------
    rng       = np.random.default_rng(RANDOM_SEED)
    all_idx   = rng.permutation(N)
    n_val     = int(np.round(VAL_FRACTION * N))
    n_train   = N - n_val
    train_idx = all_idx[:n_train]
    val_idx   = all_idx[n_train:]
    print(f"Train: {n_train}  |  Val: {n_val}")

    # Features: Y phases (N, 17)
    Y_all       = y_mod.T
    Y_train_raw = Y_all[train_idx]
    Y_val_raw   = Y_all[val_idx]

    scaler  = StandardScaler()
    Y_train = scaler.fit_transform(Y_train_raw).astype(np.float32)
    Y_val   = scaler.transform(Y_val_raw).astype(np.float32)

    # Ambiguity labels (N, NUM_AMBIGUITIES)
    A_all   = amb_diff.T
    A_train = A_all[train_idx]
    A_val   = A_all[val_idx]

    # ----------------------------------------------------------
    # 3. Build label encoders from training set
    # ----------------------------------------------------------
    encoders    = build_label_encoders(A_train.T)   # (NUM_AMBIGUITIES, n_train)
    num_classes = [enc.num_classes for enc in encoders]
    print("\nPer-branch class counts (min, max, #classes):")
    for i, enc in enumerate(encoders):
        print(f"  Branch {i+1:2d}: [{enc.amb_min:5d}, {enc.amb_max:5d}]  ->  {enc.num_classes} classes")

    C_train = np.stack([encoders[i].to_class(A_train[:, i]) for i in range(NUM_AMBIGUITIES)], axis=1)
    C_val   = np.stack([encoders[i].to_class(A_val[:,   i]) for i in range(NUM_AMBIGUITIES)], axis=1)

    # ----------------------------------------------------------
    # 4. DataLoaders
    # ----------------------------------------------------------
    def make_dataset(Y, C):
        tensors = [torch.from_numpy(Y)] + [torch.from_numpy(C[:, i]) for i in range(NUM_AMBIGUITIES)]
        return TensorDataset(*tensors)

    train_loader = DataLoader(make_dataset(Y_train, C_train),
                              batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(make_dataset(Y_val,   C_val),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # ----------------------------------------------------------
    # 5. Build model
    # ----------------------------------------------------------
    model = AmbiguityMLP(num_classes_per_branch=num_classes, dropout=DROPOUT_RATE).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel parameters: {total_params:,}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

    # ----------------------------------------------------------
    # 6. Training loop  (train loss only — no val pass per epoch)
    # ----------------------------------------------------------
    train_losses = []

    def fmt(s):
        m, sc = divmod(int(s), 60)
        return f"{m:02d}:{sc:02d}" if m < 60 else f"{m//60}h{m%60:02d}m"

    print(f"\nTraining for {NUM_EPOCHS} epochs...\n{'='*50}")
    total_start = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        epoch_start = time.time()
        tr_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        scheduler.step()
        train_losses.append(tr_loss)

        epoch_secs     = time.time() - epoch_start
        elapsed_secs   = time.time() - total_start
        remaining_secs = elapsed_secs / epoch * (NUM_EPOCHS - epoch)

        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
              f"Train loss: {tr_loss:.4f} | "
              f"Epoch: {epoch_secs:.1f}s  Elapsed: {fmt(elapsed_secs)}  ETA: {fmt(remaining_secs)}")

    # ----------------------------------------------------------
    # 7. Final validation metrics  (single pass after training)
    # ----------------------------------------------------------
    print("\n" + "="*50)
    print("FINAL VALIDATION METRICS")
    print("="*50)
    _, final_branch_acc, final_overall_acc, all_pred_classes = evaluate(
        model, val_loader, criterion, DEVICE, encoders)

    print(f"\nOverall classification accuracy : {final_overall_acc*100:.4f}%")
    print("\nPer-branch accuracy:")
    for i, acc in enumerate(final_branch_acc):
        print(f"  Branch {i+1:2d}: {acc*100:.4f}%")

    # ----------------------------------------------------------
    # 8. Positioning error using predicted ambiguities
    # ----------------------------------------------------------
    print("\n" + "="*50)
    print("POSITIONING ERROR EVALUATION ON VALIDATION SET")
    print("="*50)

    # Decode predicted classes -> ambiguity values  (NUM_AMBIGUITIES, n_val)
    pred_amb_val = np.stack([
        encoders[i].to_ambiguity(all_pred_classes[i]) for i in range(NUM_AMBIGUITIES)
    ], axis=0)

    # Full (16, n_val) ambiguity matrices for positioning
    true_amb_full = amb_diff_full[:, val_idx]           # (16, n_val)
    phase_val     = phase_diff_cycles[:, val_idx]        # (16, n_val)
    x_true_val    = x_true[:, val_idx]                   # (3,  n_val)

    # For predicted: fill first NUM_AMBIGUITIES rows with predictions,
    # keep true values for remaining rows
    mixed_amb = true_amb_full.copy()
    mixed_amb[:NUM_AMBIGUITIES, :] = pred_amb_val

    delta_r_pred, _     = combine_phase_and_ambiguity(mixed_amb,      phase_val)
    delta_r_true_amb, _ = combine_phase_and_ambiguity(true_amb_full,  phase_val)

    print("\nRunning solver with TRUE ambiguities (reference)...")
    x_hat_true  = estimate_positions_from_delta_r(delta_r_true_amb, ap_locs, x_init=x_true_val)
    errors_true = np.linalg.norm(x_hat_true - x_true_val, axis=0)

    print("\nRunning solver with PREDICTED ambiguities...")
    x_hat_pred  = estimate_positions_from_delta_r(delta_r_pred, ap_locs)
    errors_pred = np.linalg.norm(x_hat_pred - x_true_val, axis=0)

    def print_pos_metrics(label, errors):
        print(f"\n--- {label} ---")
        print(f"  Mean   : {np.mean(errors):.6f} m")
        print(f"  Median : {np.median(errors):.6f} m")
        print(f"  RMSE   : {np.sqrt(np.mean(errors**2)):.6f} m")
        print(f"  95th   : {np.percentile(errors, 95):.6f} m")
        print(f"  99th   : {np.percentile(errors, 99):.6f} m")
        print(f"  Max    : {np.max(errors):.6f} m")
        for thr_m, lbl in [(0.01,"1cm"),(0.05,"5cm"),(0.10,"10cm"),(0.50,"50cm"),(1.0,"1m")]:
            print(f"  <= {lbl:3s}  : {100*np.mean(errors<=thr_m):.2f}%")

    print_pos_metrics("True ambiguities (reference)", errors_true)
    print_pos_metrics("Predicted ambiguities",        errors_pred)

    # ----------------------------------------------------------
    # 9. Plots
    # ----------------------------------------------------------
    plot_training_curves(train_losses, out_dir)
    plot_branch_accuracy(final_branch_acc, out_dir)
    plot_position_error_cdf(errors_true, errors_pred, out_dir)

    # ----------------------------------------------------------
    # 10. Save model + scaler
    # ----------------------------------------------------------
    model_path  = out_dir / MODEL_FILE
    scaler_path = out_dir / SCALER_FILE

    torch.save({
        "model_state_dict":       model.state_dict(),
        "num_classes_per_branch": num_classes,
        "encoders_min":           [enc.amb_min for enc in encoders],
        "encoders_max":           [enc.amb_max for enc in encoders],
        "num_ambiguities":        NUM_AMBIGUITIES,
        "dropout":                DROPOUT_RATE,
        "freq_hz":                FREQ,
        "c":                      C,
        "final_val_acc":          final_overall_acc,
        "branch_val_acc":         final_branch_acc.tolist(),
    }, str(model_path))
    print(f"\nSaved model      -> {model_path}")

    joblib.dump(scaler, str(scaler_path))
    print(f"Saved scaler     -> {scaler_path}")

    zip_path = out_dir / ZIP_FILE
    with zipfile.ZipFile(str(zip_path), "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(str(model_path),  MODEL_FILE)
        zf.write(str(scaler_path), SCALER_FILE)
    print(f"Saved zip        -> {zip_path}")

    try:
        from google.colab import files
        files.download(str(zip_path))
        print("Download triggered in Colab.")
    except ImportError:
        print("(Not running in Colab — download the zip manually from the file browser.)")

    # ----------------------------------------------------------
    # 11. Save numerical results to .mat
    # ----------------------------------------------------------
    savemat(str(out_dir / "classifier_results.mat"), {
        "val_idx":              val_idx,
        "train_idx":            train_idx,
        "pred_amb_val":         pred_amb_val,
        "true_amb_full_val":    true_amb_full,
        "x_hat_pred":           x_hat_pred,
        "x_hat_true_ref":       x_hat_true,
        "x_true_val":           x_true_val,
        "errors_pred":          errors_pred,
        "errors_true_ref":      errors_true,
        "branch_val_acc":       final_branch_acc,
        "overall_val_acc":      final_overall_acc,
        "num_ambiguities":      NUM_AMBIGUITIES,
    })
    print(f"Saved .mat results -> {out_dir / 'classifier_results.mat'}")
    print("\nDone.")


if __name__ == "__main__":
    main()

Using device: cuda
Loading data...
Total samples in files: 1250000
Subsampled to 1000000 samples (NUM_SAMPLES=1000000)

Estimating 16/16 ambiguities
Train: 990000  |  Val: 10000

Per-branch class counts (min, max, #classes):
  Branch  1: [   -6,     6]  ->  13 classes
  Branch  2: [   -9,     9]  ->  19 classes
  Branch  3: [   -7,     7]  ->  15 classes
  Branch  4: [   -6,     6]  ->  13 classes
  Branch  5: [   -6,     6]  ->  13 classes
  Branch  6: [   -9,     9]  ->  19 classes
  Branch  7: [   -9,     9]  ->  19 classes
  Branch  8: [   -8,     8]  ->  17 classes
  Branch  9: [  -12,    12]  ->  25 classes
  Branch 10: [  -12,    12]  ->  25 classes
  Branch 11: [  -12,    13]  ->  26 classes
  Branch 12: [  -11,    11]  ->  23 classes
  Branch 13: [  -11,    12]  ->  24 classes
  Branch 14: [  -15,    16]  ->  32 classes
  Branch 15: [  -14,    14]  ->  29 classes
  Branch 16: [  -10,    10]  ->  21 classes

Model parameters: 3,873,741

Training for 1 epochs...
Epoch   1/1 | Tr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download triggered in Colab.
Saved .mat results -> classifier_results.mat

Done.
